# Snæfellsjökull: annual satellite imagery

This notebook uses **Google Earth Engine** to build **one cloud-masked, late-summer composite per year** of Snæfellsjökull, the ice cap on the tip of the Snæfellsnes peninsula in western Iceland (summit ≈ 64.808° N, 23.774° W, 1,446 m). It then:

1. Shows the composites on an interactive map with a year slider
2. Downloads each year as true-colour and false-colour PNGs, and as a multi-band GeoTIFF
3. Plots grids of the annual images
4. Estimates the late-summer snow and ice area each year
5. Writes an animated GIF timelapse, with every frame also saved as a PNG

**Data sources**
- **Landsat 5 / 7 / 8 / 9**, Collection 2 Level-2 surface reflectance, 30 m, 1984 to present
- **Sentinel-2** L2A surface reflectance (harmonized), 10–20 m, 2017 to present, masked with Cloud Score+
- **GLIMS** glacier outlines (about 2002–2004), used as the area for the snow and ice statistics

Composites cover **July–September**, near the end of the melt season, when seasonal snow cover is smallest and the glacier is easiest to see. Iceland is very cloudy, so some years have few clear scenes.

## Outputs

Everything is written to `outputs/` next to this notebook. `<sensor>` is `landsat` or `sentinel2`.

| Path | Contents |
|---|---|
| `outputs/rgb/snaefellsjokull_<sensor>_<year>.png` | True-colour image of each year's composite (red/green/blue) |
| `outputs/false_color/snaefellsjokull_<sensor>_<year>.png` | False-colour image (SWIR1/NIR/green): snow and ice show as cyan, cloud as white |
| `outputs/geotiff/snaefellsjokull_<sensor>_<year>.tif` | Composite as a 7-band GeoTIFF, UTM zone 27N (EPSG:32627): `blue, green, red, nir, swir1, swir2, ndsi`. Values are reflectance (or NDSI) × 10,000 as 16-bit integers; no-data is `-32768`. |
| `outputs/snaefellsjokull_<sensor>_grid_rgb.png`, `..._grid_false_color.png` | All years side by side |
| `outputs/snaefellsjokull_<sensor>_snow_ice_area.csv` / `.png` | Snow and ice area per year, as a table and a chart |
| `outputs/snaefellsjokull_<sensor>_timelapse.gif` | Animated timelapse |
| `outputs/snaefellsjokull_<sensor>_timelapse_frames/1.png, 2.png, …` | Every GIF frame as a PNG, in year order |

Set `SAVE_GEOTIFF = False` to skip the GeoTIFFs (about 2 MB per year for Landsat, up to about 18 MB for Sentinel-2).

### Figure style

All figures use one minimal black-and-white style: white background, black text and lines, greys for secondary elements, no gridlines and no top or right axis borders. Satellite images keep their natural colours, because the colour is the data.

## 1. Setup

In [ ]:
# Install dependencies (run once)
%pip install -q earthengine-api geemap pandas numpy matplotlib pillow requests

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import ee
import geemap
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from IPython.display import Image as ShowImage
from PIL import Image

In [ ]:
from cycler import cycler

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
    "text.color": "black",
    "axes.edgecolor": "black",
    "axes.labelcolor": "black",
    "xtick.color": "black",
    "ytick.color": "black",
    "axes.prop_cycle": cycler(color=["black"]),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": False,
    "axes.linewidth": 0.8,
    "axes.titlesize": 11,
    "axes.titlelocation": "left",
    "font.family": "sans-serif",
    "font.size": 10,
    "lines.linewidth": 1.2,
    "legend.frameon": False,
})

GREY = "#9a9a9a"
LIGHT_GREY = "#d4d4d4"

In [ ]:
# Authenticate and initialize Earth Engine.
# Use a Google Cloud project that has the Earth Engine API enabled.
PROJECT = "gsapp-map"

try:
    ee.Initialize(project=PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=PROJECT)

print("Earth Engine initialized")

## 2. Settings

In [ ]:
# Snæfellsjökull summit
SUMMIT = ee.Geometry.Point([-23.774, 64.808])

# Area of interest: roughly 11 x 11 km box around the ice cap
AOI = ee.Geometry.Rectangle([-23.90, 64.76, -23.65, 64.86])

# 'landsat' (1984 to present, 30 m) or 'sentinel2' (2017 to present, 10 m)
SENSOR = "landsat"

START_YEAR = 1985 if SENSOR == "landsat" else 2017
END_YEAR = 2025

# Late-summer window (calendar months, inclusive)
START_MONTH, END_MONTH = 7, 9

# Skip scenes whose whole-scene cloud cover exceeds this percentage
MAX_SCENE_CLOUD = 70

# NDSI threshold for snow and ice classification
NDSI_THRESHOLD = 0.4

SCALE = 30 if SENSOR == "landsat" else 10
EXPORT_CRS = "EPSG:32627"  # UTM zone 27N

# Outputs
OUT_DIR = Path("outputs")
RGB_DIR = OUT_DIR / "rgb"
FALSE_COLOR_DIR = OUT_DIR / "false_color"
GEOTIFF_DIR = OUT_DIR / "geotiff"
NAME = f"snaefellsjokull_{SENSOR}"

SAVE_GEOTIFF = True
IMAGE_PX = 600            # longest side of the PNG images, in pixels
GEOTIFF_NODATA = -32768
DOWNLOAD_THREADS = 6      # parallel downloads from Earth Engine

GIF_FRAME_MS = 600
GIF_LAST_FRAME_MS = 2000

for folder in (RGB_DIR, FALSE_COLOR_DIR, GEOTIFF_DIR):
    folder.mkdir(parents=True, exist_ok=True)

## 3. Glacier outline (GLIMS)

GLIMS holds outlines of Snæfellsjökull from 2002 and 2004. They are merged and buffered by 500 m to make the region for the snow and ice statistics, so it also covers ice in earlier, larger years. If GLIMS returns nothing, a 4.5 km circle around the summit is used instead.

For dated outlines from ~1890 to 2025, see `../Snæfellsjökull_outlines/download_outlines.ipynb`.

In [ ]:
glims = ee.FeatureCollection("GLIMS/20230607").filterBounds(AOI)
n_outlines = glims.size().getInfo()
print(f"GLIMS outlines in AOI: {n_outlines}")

if n_outlines > 0:
    glacier_outline = glims.geometry().dissolve(maxError=10)
else:
    glacier_outline = SUMMIT.buffer(4500)

# Region used for snow and ice area statistics
STATS_REGION = glacier_outline.buffer(500).intersection(AOI, maxError=10)

## 4. Image collections and cloud masking

All sensors are renamed to common band names (`blue, green, red, nir, swir1, swir2`) and scaled to surface reflectance (0–1), so the rest of the notebook works the same for every sensor.

- **Clouds:** Landsat clouds and shadows are removed with its built-in quality flags (`QA_PIXEL`). Pixels flagged as snow are kept. Sentinel-2 clouds are removed with Google's Cloud Score+.
- **Saturated pixels are kept.** Landsat 5 and 7 often saturate over bright snow, and removing those pixels would remove most of the ice cap.
- **Landsat 7** has striped gaps since its Scan Line Corrector failed in May 2003. It is still used for 1999–2012, because Landsat 5 has almost no scenes of this area in 2003–2011 and none in 2012. The median composite partly fills the stripes. From 2013, Landsat 8 and 9 replace it.

In [ ]:
COMMON_BANDS = ["blue", "green", "red", "nir", "swir1", "swir2"]


def mask_landsat(img):
    """Apply Collection 2 scale factors and mask clouds and shadows with QA_PIXEL."""
    qa = img.select("QA_PIXEL")
    # Bit 1: dilated cloud, bit 3: cloud, bit 4: cloud shadow.
    # Bit 5 (snow) is deliberately NOT masked.
    clear = (
        qa.bitwiseAnd(1 << 1).eq(0)
        .And(qa.bitwiseAnd(1 << 3).eq(0))
        .And(qa.bitwiseAnd(1 << 4).eq(0))
    )
    optical = img.select("SR_B.").multiply(0.0000275).add(-0.2)
    # copyProperties returns a generic ee.Element, so cast back to ee.Image
    return ee.Image(
        optical.updateMask(clear).copyProperties(
            img, ["system:time_start", "CLOUD_COVER", "SPACECRAFT_ID"]
        )
    )


def prep_l457(img):
    return mask_landsat(img).select(
        ["SR_B1", "SR_B2", "SR_B3", "SR_B4", "SR_B5", "SR_B7"], COMMON_BANDS
    )


def prep_l89(img):
    return mask_landsat(img).select(
        ["SR_B2", "SR_B3", "SR_B4", "SR_B5", "SR_B6", "SR_B7"], COMMON_BANDS
    )


def landsat_collection():
    def base(coll_id):
        return (
            ee.ImageCollection(coll_id)
            .filterBounds(AOI)
            .filter(ee.Filter.lt("CLOUD_COVER", MAX_SCENE_CLOUD))
        )

    l5 = base("LANDSAT/LT05/C02/T1_L2").map(prep_l457)
    l7 = (
        base("LANDSAT/LE07/C02/T1_L2")
        .filter(ee.Filter.date("1999-01-01", "2013-01-01"))
        .map(prep_l457)
    )
    l8 = base("LANDSAT/LC08/C02/T1_L2").map(prep_l89)
    l9 = base("LANDSAT/LC09/C02/T1_L2").map(prep_l89)
    return l5.merge(l7).merge(l8).merge(l9)


def sentinel2_collection():
    cs_plus = ee.ImageCollection("GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED")

    def prep_s2(img):
        clear = img.select("cs_cdf").gte(0.6)
        return ee.Image(
            img.select(["B2", "B3", "B4", "B8", "B11", "B12"], COMMON_BANDS)
            .divide(10000)
            .updateMask(clear)
            .copyProperties(img, ["system:time_start", "CLOUDY_PIXEL_PERCENTAGE"])
        )

    return (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(AOI)
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", MAX_SCENE_CLOUD))
        .linkCollection(cs_plus, ["cs_cdf"])
        .map(prep_s2)
    )


SOURCE = landsat_collection() if SENSOR == "landsat" else sentinel2_collection()

## 5. Annual composites

In [ ]:
def scenes_for_year(year):
    return (
        SOURCE
        .filter(ee.Filter.calendarRange(year, year, "year"))
        .filter(ee.Filter.calendarRange(START_MONTH, END_MONTH, "month"))
    )


def annual_composite(year):
    """Median late-summer composite for one year, with an NDSI band added."""
    scenes = scenes_for_year(year)
    comp = scenes.median().clip(AOI)
    ndsi = comp.normalizedDifference(["green", "swir1"]).rename("ndsi")
    return (
        comp.addBands(ndsi)
        .set("year", year)
        .set("n_scenes", scenes.size())
        .set("system:time_start", ee.Date.fromYMD(year, 8, 15).millis())
    )


# Count scenes per year in a single server call, then skip empty years
years = list(range(START_YEAR, END_YEAR + 1))
counts = ee.List(years).map(lambda y: scenes_for_year(ee.Number(y)).size()).getInfo()
scene_counts = dict(zip(years, counts))

valid_years = [y for y, n in scene_counts.items() if n > 0]
missing = [y for y, n in scene_counts.items() if n == 0]

print("Scenes per year:", scene_counts)
print(f"\n{len(valid_years)} years with imagery; no imagery for: {missing or 'none'}")

composites = ee.ImageCollection([annual_composite(y) for y in valid_years])

## 6. Interactive map with a year slider

- **True colour** (red/green/blue): what the eye sees
- **False colour** (SWIR1/NIR/green): snow and ice show as bright cyan, cloud as white or pink, rock and vegetation as brown and green. This makes it easier to tell ice from cloud.

The GLIMS outline is drawn in white.

In [ ]:
TRUE_COLOR = {"bands": ["red", "green", "blue"], "min": 0.0, "max": 0.9, "gamma": 1.4}
FALSE_COLOR = {"bands": ["swir1", "nir", "green"], "min": 0.0, "max": 0.8, "gamma": 1.2}

Map = geemap.Map(center=[64.808, -23.774], zoom=12, basemap="Esri.WorldImagery")
Map.add_time_slider(
    composites,
    TRUE_COLOR,
    labels=[str(y) for y in valid_years],
    time_interval=1,
    layer_name="Annual composite",
)
Map.addLayer(
    ee.FeatureCollection(glims).style(color="white", fillColor="00000000", width=1),
    {},
    "GLIMS outlines",
)
Map

## 7. Download the annual images

For each year this downloads the true-colour PNG, the false-colour PNG and (if `SAVE_GEOTIFF`) the 7-band GeoTIFF. Downloads run in parallel. Areas with no cloud-free data are transparent in the PNGs and `-32768` in the GeoTIFFs.

In [ ]:
def composite_for(year):
    return ee.Image(composites.filter(ee.Filter.eq("year", year)).first())


def fetch(url, path):
    r = requests.get(url, timeout=300)
    r.raise_for_status()
    path.write_bytes(r.content)


def download_png(img, vis, path):
    url = img.visualize(**vis).getThumbURL(
        {"region": AOI, "dimensions": IMAGE_PX, "format": "png", "crs": "EPSG:3857"}
    )
    fetch(url, path)


def download_geotiff(img, path):
    scaled = (
        img.select(COMMON_BANDS + ["ndsi"])
        .multiply(10000).round().toInt16()
        .unmask(GEOTIFF_NODATA, False)
    )
    url = scaled.getDownloadURL(
        {"format": "GEO_TIFF", "region": AOI, "scale": SCALE, "crs": EXPORT_CRS}
    )
    fetch(url, path)


def download_year(year):
    img = composite_for(year)
    download_png(img, TRUE_COLOR, RGB_DIR / f"{NAME}_{year}.png")
    download_png(img, FALSE_COLOR, FALSE_COLOR_DIR / f"{NAME}_{year}.png")
    if SAVE_GEOTIFF:
        download_geotiff(img, GEOTIFF_DIR / f"{NAME}_{year}.tif")
    return year


with ThreadPoolExecutor(max_workers=DOWNLOAD_THREADS) as pool:
    for year in pool.map(download_year, valid_years):
        print("downloaded", year)

print(f"\nPNGs in {RGB_DIR}/ and {FALSE_COLOR_DIR}/" + (f", GeoTIFFs in {GEOTIFF_DIR}/" if SAVE_GEOTIFF else ""))

## 8. Grids of the annual images

In [ ]:
def load_on_white(path):
    """Open a PNG and flatten its transparent (no-data) areas onto white."""
    img = Image.open(path).convert("RGBA")
    background = Image.new("RGBA", img.size, "white")
    background.alpha_composite(img)
    return background.convert("RGB")


def plot_grid(folder, title, out_path, ncols=6):
    nrows = -(-len(valid_years) // ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 2.2, nrows * 2.3))
    for ax in axes.flat:
        ax.axis("off")
    for ax, year in zip(axes.flat, valid_years):
        ax.imshow(load_on_white(folder / f"{NAME}_{year}.png"))
        ax.set_title(str(year), fontsize=9)
    fig.suptitle(title, x=0.01, ha="left", fontsize=12)
    fig.tight_layout()
    fig.savefig(out_path, dpi=150)
    plt.show()


plot_grid(RGB_DIR, f"Snæfellsjökull, {START_MONTH}–{END_MONTH} median composites ({SENSOR})",
          OUT_DIR / f"{NAME}_grid_rgb.png")

In [ ]:
plot_grid(FALSE_COLOR_DIR, "False colour (SWIR1/NIR/green): snow and ice show as cyan",
          OUT_DIR / f"{NAME}_grid_false_color.png")

## 9. Late-summer snow and ice area

Pixels with **NDSI > 0.4** and **NIR > 0.11** (to exclude dark water) are classed as snow or ice, and the area is summed inside the statistics region.

`valid_fraction` is the share of the region with cloud-free data in that year's composite. Years below 95% are drawn as hollow grey points, because gaps from cloud masking make the area too low.

> This measures *late-summer snow and ice cover*, not the glacier boundary itself. Snowy summers inflate it, so it runs above the mapped glacier area (e.g. 13.4 km² vs 8.6 km² in 2021). Use it as a trend, not as a glacier outline.

In [ ]:
def snow_stats(img):
    snow = (
        img.select("ndsi").gt(NDSI_THRESHOLD)
        .And(img.select("nir").gt(0.11))
        .rename("snow")
    )
    area_km2 = (
        snow.multiply(ee.Image.pixelArea()).divide(1e6)
        .reduceRegion(ee.Reducer.sum(), STATS_REGION, scale=SCALE, maxPixels=1e9)
        .get("snow")
    )
    valid = (
        img.select("green").mask().rename("valid")
        .reduceRegion(ee.Reducer.mean(), STATS_REGION, scale=SCALE, maxPixels=1e9)
        .get("valid")
    )
    return ee.Feature(None, {
        "year": img.get("year"),
        "n_scenes": img.get("n_scenes"),
        "snow_ice_km2": area_km2,
        "valid_fraction": valid,
    })


stats = ee.FeatureCollection(composites.map(snow_stats)).getInfo()
df = pd.DataFrame([f["properties"] for f in stats["features"]]).set_index("year").sort_index()
df["reliable"] = df["valid_fraction"] >= 0.95
df.to_csv(OUT_DIR / f"{NAME}_snow_ice_area.csv")
df.round(3)

In [ ]:
good, bad = df[df["reliable"]], df[~df["reliable"]]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(good.index, good["snow_ice_km2"], "o-", color="black", markersize=4,
        label="≥95% cloud-free coverage")
ax.plot(bad.index, bad["snow_ice_km2"], "o", markersize=4, markerfacecolor="white",
        markeredgecolor=GREY, label="<95% cloud-free coverage (unreliable)")
ax.set_xlabel("Year")
ax.set_ylabel("Snow and ice area (km²)")
ax.set_ylim(bottom=0)
ax.set_title(f"Snæfellsjökull late-summer snow and ice area ({SENSOR}, NDSI > {NDSI_THRESHOLD})")
ax.legend(loc="lower left")
fig.tight_layout()
fig.savefig(OUT_DIR / f"{NAME}_snow_ice_area.png", dpi=150)
plt.show()

## 10. Animated timelapse (GIF)

Each frame is one year's true-colour image with a caption underneath. Frames are built locally from the PNGs downloaded in section 7, so the GIF and the saved frames are identical. Every frame is also saved as `outputs/<gif name>_frames/<n>.png`, numbered from 1 in year order.

In [ ]:
def figure_to_image(fig):
    """Render a matplotlib figure to a PIL image and close it."""
    fig.canvas.draw()
    image = Image.fromarray(np.asarray(fig.canvas.buffer_rgba())).convert("RGB")
    plt.close(fig)
    return image


def save_gif(frames, gif_path, durations_ms):
    """Write an animated GIF, and every frame as a PNG in <gif name>_frames/ (1.png, 2.png, ...)."""
    frames_dir = gif_path.with_name(gif_path.stem + "_frames")
    frames_dir.mkdir(parents=True, exist_ok=True)
    for old in frames_dir.glob("*.png"):
        old.unlink()
    for i, frame in enumerate(frames, start=1):
        frame.save(frames_dir / f"{i}.png")

    palette_frames = [f.convert("P", palette=Image.ADAPTIVE, colors=256) for f in frames]
    palette_frames[0].save(gif_path, save_all=True, append_images=palette_frames[1:],
                           duration=durations_ms, loop=0, optimize=True)
    print(f"wrote {gif_path} ({len(frames)} frames, {gif_path.stat().st_size / 1e6:.1f} MB)")
    print(f"wrote {len(frames)} frames to {frames_dir}/")

In [ ]:
def timelapse_frame(year):
    img = load_on_white(RGB_DIR / f"{NAME}_{year}.png")
    fig = plt.figure(figsize=(6, 6.6), dpi=100)
    ax = fig.add_axes([0.03, 0.10, 0.94, 0.87])
    ax.imshow(img)
    ax.axis("off")
    fig.text(0.04, 0.05, str(year), fontsize=20, fontweight="bold", va="center")
    fig.text(0.96, 0.05,
             f"{SENSOR.title()} · median of {scene_counts[year]} scenes, months {START_MONTH}–{END_MONTH}",
             fontsize=8, color=GREY, ha="right", va="center")
    return figure_to_image(fig)


frames = [timelapse_frame(y) for y in valid_years]
durations = [GIF_FRAME_MS] * (len(frames) - 1) + [GIF_LAST_FRAME_MS]
gif_path = OUT_DIR / f"{NAME}_timelapse.gif"
save_gif(frames, gif_path, durations)
ShowImage(filename=str(gif_path))

## 11. (Optional) Export to Google Drive

The GeoTIFFs in section 7 are downloaded directly. For a much larger area, the direct download limit (about 48 MB per image) may be exceeded. Then set `EXPORT_TO_DRIVE = True` to export each year to Google Drive instead. Track progress at https://code.earthengine.google.com/tasks.

In [ ]:
EXPORT_TO_DRIVE = False
DRIVE_FOLDER = "snaefellsjokull"

if EXPORT_TO_DRIVE:
    for year in valid_years:
        task = ee.batch.Export.image.toDrive(
            image=composite_for(year).select(COMMON_BANDS + ["ndsi"]).toFloat(),
            description=f"{NAME}_{year}",
            folder=DRIVE_FOLDER,
            region=AOI,
            scale=SCALE,
            crs=EXPORT_CRS,
            maxPixels=1e9,
        )
        task.start()
        print(f"Started export for {year}")

## Notes and caveats

- **Sensor consistency:** surface reflectance from Landsat 5/7 (TM/ETM+) and Landsat 8/9 (OLI) is close but not identical. For careful trend work, consider a cross-sensor harmonization such as Roy et al. (2016).
- **Clouds and snow:** Landsat's cloud mask sometimes labels bright snow as cloud, which leaves holes over the ice cap. The median composite and the `valid_fraction` check reduce this. If a year looks patchy, widen the month window (e.g. `START_MONTH = 6`, `END_MONTH = 10`).
- **White patches and stripes** in the images are pixels masked as cloud (or cloud shadow) in *every* scene of that summer, which are transparent and shown on white. The blocky rectangles are Landsat's dilated-cloud flag; the diagonal stripes in 2003–2012 are Landsat 7's failed Scan Line Corrector. For gap-free images, use `v1_satellite_clearest_scene.ipynb`, which takes the single clearest day instead of a masked median.
- **Cloudy years:** 1997, 2001, 2005, 2010 and 2018 have little cloud-free data over the glacier in July–September.
- **Sentinel-2:** set `SENSOR = "sentinel2"` for sharper 10 m composites from 2017 onward. File names change accordingly, so both sensors' outputs can sit side by side.